<a href="https://colab.research.google.com/github/mas622424/WISER-BQP-QAPINN/blob/main/notebooks/01_cPINN_Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
import torch.nn as nn
import numpy as np

# 1. Same Physics Residual Function
def compute_burgers_residual(model, x, t, nu=0.01/np.pi):
    x.requires_grad_(True)
    t.requires_grad_(True)
    u = model(torch.cat([x, t], dim=1))
    u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), retain_graph=True, create_graph=True)[0]
    u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), retain_graph=True, create_graph=True)[0]
    u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), retain_graph=True, create_graph=True)[0]
    return u_t + u * u_x - nu * u_xx

# 2. Classical PINN Architecture (No Quantum Layer)
class ClassicalPINN(nn.Module):
    def __init__(self):
        super(ClassicalPINN, self).__init__()
        # We replace the 4-qubit quantum layer with a classical layer of the same size
        self.input_layer = nn.Linear(2, 4)
        self.classical_hidden = nn.Linear(4, 4) # The replacement layer
        self.hidden1 = nn.Linear(4, 20)
        self.act1 = nn.Tanh()
        self.hidden2 = nn.Linear(20, 20)
        self.act2 = nn.Tanh()
        self.output_layer = nn.Linear(20, 1)

    def forward(self, x_t):
        out = torch.tanh(self.input_layer(x_t))
        out = torch.tanh(self.classical_hidden(out)) # Classical operation instead of quantum
        out = self.act1(self.hidden1(out))
        out = self.act2(self.hidden2(out))
        return self.output_layer(out)

# 3. Initialize Model and Print Parameter Count
cpinn_model = ClassicalPINN()
total_params = sum(p.numel() for p in cpinn_model.parameters() if p.requires_grad)
print(f"Total Trainable Parameters in Classical PINN: {total_params}")

Total Trainable Parameters in Classical PINN: 573


The Classical Training Loop

In [6]:
def train_cpinn(model, epochs=1000, lr=0.01):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    mse_loss = nn.MSELoss()

    loss_history = []

    print("Starting Classical PINN Training...")

    for epoch in range(epochs):
        optimizer.zero_grad()

        # 1. Collocation points inside the domain (for PDE residual)
        x_col = torch.rand(100, 1) * 2 - 1  # x in [-1, 1]
        t_col = torch.rand(100, 1)          # t in [0, 1]
        f_res = compute_burgers_residual(model, x_col, t_col)
        loss_pde = torch.mean(f_res ** 2)

        # 2. Initial Condition: u(x, 0) = -sin(pi * x)
        x_ic = torch.rand(50, 1) * 2 - 1
        t_ic = torch.zeros(50, 1)
        u_ic_pred = model(torch.cat([x_ic, t_ic], dim=1))
        u_ic_true = -torch.sin(np.pi * x_ic)
        loss_ic = mse_loss(u_ic_pred, u_ic_true)

        # 3. Boundary Condition: u(-1, t) = u(1, t) = 0
        t_bc = torch.rand(50, 1)
        x_bc_left = -torch.ones(50, 1)
        x_bc_right = torch.ones(50, 1)
        u_bc_left = model(torch.cat([x_bc_left, t_bc], dim=1))
        u_bc_right = model(torch.cat([x_bc_right, t_bc], dim=1))
        loss_bc = torch.mean(u_bc_left ** 2) + torch.mean(u_bc_right ** 2)

        # Total Physics-Informed Loss
        total_loss = loss_pde + loss_ic + loss_bc
        total_loss.backward()
        optimizer.step()

        # Record the loss for Role 3
        loss_history.append(total_loss.item())

        if epoch % 100 == 0:
            print(f"Epoch {epoch:4d} | Total Loss: {total_loss.item():.5f} | PDE Loss: {loss_pde.item():.5f}")

    return loss_history

# Train the classical model!
loss_history = train_cpinn(cpinn_model, epochs=1000, lr=0.01)

Starting Classical PINN Training...
Epoch    0 | Total Loss: 0.51069 | PDE Loss: 0.00009
Epoch  100 | Total Loss: 0.28638 | PDE Loss: 0.02828
Epoch  200 | Total Loss: 0.18605 | PDE Loss: 0.04873
Epoch  300 | Total Loss: 0.15664 | PDE Loss: 0.05821
Epoch  400 | Total Loss: 0.13790 | PDE Loss: 0.05575
Epoch  500 | Total Loss: 0.15218 | PDE Loss: 0.05007
Epoch  600 | Total Loss: 0.12576 | PDE Loss: 0.03511
Epoch  700 | Total Loss: 0.12537 | PDE Loss: 0.04625
Epoch  800 | Total Loss: 0.12310 | PDE Loss: 0.05400
Epoch  900 | Total Loss: 0.10370 | PDE Loss: 0.03045


In [8]:
import numpy as np

print("Generating 256-point evaluation grid and extracting artifacts for Role 3 (cPINN)...")

# 1. Create a 256x100 grid to match the ground_truth.npy dimensions exactly
x_grid = torch.linspace(-1, 1, 256)
t_grid = torch.linspace(0, 1, 100)
X, T = torch.meshgrid(x_grid, t_grid, indexing="ij")
inputs = torch.cat([X.reshape(-1, 1), T.reshape(-1, 1)], dim=1)

# 2. Setup PyTorch Hooks to capture internal Layer 2 and Layer 3 activations
activations = {}
def get_activation(name):
    def hook(model, input, output):
        activations[name] = output.detach().numpy()
    return hook

# Attach hooks to the Tanh activations of the hidden layers in the classical model
# NOTE: Make sure the layer names match exactly what you defined in the ClassicalPINN class
h1 = cpinn_model.act1.register_forward_hook(get_activation('layer2'))
h2 = cpinn_model.act2.register_forward_hook(get_activation('layer3'))

# 3. Run the grid through the trained classical model
with torch.no_grad():
    u_pred_flat = cpinn_model(inputs).numpy()

# Remove hooks
h1.remove()
h2.remove()

# 4. Format and Save Predictions (shape: 256x100)
u_pred = u_pred_flat.reshape(256, 100)
np.save('predictions_cpinn.npy', u_pred)

# 5. Format and Save Activations (shape: 20 neurons, 256x100 grid)
act_layer2 = activations['layer2'].reshape(256, 100, 20).transpose(2, 0, 1)
act_layer3 = activations['layer3'].reshape(256, 100, 20).transpose(2, 0, 1)
np.savez('activations_cpinn.npz', layer2=act_layer2, layer3=act_layer3)

# 6. Save Loss Curve (Assuming your training loop variable is called `loss_history`)
np.save('loss_curve_cpinn.npy', np.array(loss_history))

# 7. Save Flattened Weights
weights_list = [p.detach().numpy().flatten() for p in cpinn_model.parameters()]
flat_weights = np.concatenate(weights_list)
np.savez('weights_cpinn.npz', weights=flat_weights)

# 8. Print Summary
total_params = sum(p.numel() for p in cpinn_model.parameters() if p.requires_grad)
print("\n CLASSICAL PINN EXPORT COMPLETE!")
print(f"Total Trainable Parameters: {total_params}")

Generating 256-point evaluation grid and extracting artifacts for Role 3 (cPINN)...

 CLASSICAL PINN EXPORT COMPLETE!
Total Trainable Parameters: 573
